In [ ]:
!pip install -q roboflow

# Store your key in Colab: left sidebar > Secrets (key icon) > add ROBOFLOW_API_KEY
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace("site-construction-safety").project("site-construction-safety")
version = project.version(1)
dataset = version.download("yolov11")


In [11]:
!cat {dataset.location}/data.yaml
!ls {dataset.location}/train/images | wc -l
!ls {dataset.location}/valid/images | wc -l
!ls {dataset.location}/test/images | wc -l


train: ../train/images
val: ../valid/images
test: ../test/images

nc: 7
names: ['Gloves', 'Helmet', 'Non-Helmet', 'Person', 'Shoes', 'Vest', 'bare-arms']

roboflow:
  workspace: site-construction-safety
  project: site-construction-safety
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/site-construction-safety/site-construction-safety/dataset/19861
340
231


In [3]:
# 1. Install (needed every new session)
!pip install -q ultralytics roboflow

In [4]:
import ultralytics
ultralytics.checks()
from ultralytics import YOLO


Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 0.1/112.6 GB disk)


In [7]:
# 3. Mount Drive (your training results live here)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import zipfile

# Extract the dataset if it's still zipped
zip_path = f"{dataset.location}/roboflow.zip"
if os.path.exists(zip_path) and not os.path.exists(f"{dataset.location}/data.yaml"):
    print(f"Extracting {zip_path} to {dataset.location}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset.location)
    print("Extraction complete!")

model = YOLO("yolo11n.pt")
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=40, imgsz=640, batch=16, patience=10,
    project="/content/drive/MyDrive/cv-runs",
    name="ppe_yolo11n_e40",
)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Site-Construction-Safety-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ppe_yolo11n_e40-4, nbs=64, nms=None, opset=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e886b265160>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
  

In [12]:
model_n = YOLO("/content/drive/MyDrive/cv-runs/ppe_yolo11n_e40-4/weights/best.pt")
test_n = model_n.val(data=f"{dataset.location}/data.yaml", split="test")

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,598,509 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 707.3±437.0 MB/s, size: 39.2 KB)
val: Scanning /content/Site-Construction-Safety-1/test/labels... 231 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 231/231 1.5Kit/s 0.2s
val: New cache created: /content/Site-Construction-Safety-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.9it/s 5.1s
                   all        231       2519      0.955      0.907      0.955      0.747
                Gloves        188        510      0.969      0.912       0.96      0.698
                Helmet        173        284      0.974      0.937      0.987      0.753
            Non-Helmet         55         71      0.902      0.915      0.961       0.75
                Person        217        626  

In [ ]:
model_s = YOLO("yolo11s.pt")
model_s.train(
    data=f"{dataset.location}/data.yaml",
    epochs=20, imgsz=640, batch=16, patience=5,
    project="/content/drive/MyDrive/cv-runs",
    name="ppe_yolo11s_e20",
)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Site-Construction-Safety-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ppe_yolo11s_e20, nbs=64, nms=None, opset=No

In [15]:
model_s = YOLO("/content/drive/MyDrive/cv-runs/ppe_yolo11s_e20/weights/best.pt")
test_s = model_s.val(
    data=f"{dataset.location}/data.yaml",
    split="test",
    project="/content/drive/MyDrive/cv-runs",
    name="ppe_yolo11s_test",
)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,415,509 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 182.1±362.4 MB/s, size: 37.7 KB)
val: Scanning /content/Site-Construction-Safety-1/test/labels.cache... 231 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 231/231 69.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 3.1it/s 4.9s
                   all        231       2519      0.958       0.91      0.963      0.774
                Gloves        188        510      0.974      0.922      0.972      0.727
                Helmet        173        284      0.977      0.947       0.98      0.769
            Non-Helmet         55         71       0.95      0.944      0.976      0.796
                Person        217        626      0.961      0.943      0.979      0.855
                 Shoes     